In [2]:
from pathlib import Path
import pandas as pd

data_file = Path(r"D:\9011_py_project\ds-ml-case-studies\data-exploration\02-beer-data-analysis\data\BeerDataScienceProject.tar.bz2")
print("cwd:", Path.cwd())
print("file exists:", data_file.exists())

df = pd.read_csv(data_file, compression="bz2", encoding="latin-1")
print(df.shape)
print(df.columns.tolist())
df.head()

cwd: D:\9011_py_project\ds-ml-case-studies
file exists: True
(528870, 13)
['beer_ABV', 'beer_beerId', 'beer_brewerId', 'beer_name', 'beer_style', 'review_appearance', 'review_palette', 'review_overall', 'review_taste', 'review_profileName', 'review_aroma', 'review_text', 'review_time']


,beer_ABV,beer_beerId,beer_brewerId,beer_name,beer_style,review_appearance,review_palette,review_overall,review_taste,review_profileName,review_aroma,review_text,review_time
0,5.0,47986,10325,Sausa Weizen,Hefeweizen,2.5,2.0,1.5,1.5,stcules,1.5,A lot of foam. But a lot. In the smell some ba...,1234817823
1,6.2,48213,10325,Red Moon,English Strong Ale,3.0,2.5,3.0,3.0,stcules,3.0,"Dark red color, light beige foam, average. In ...",1235915097
2,6.5,48215,10325,Black Horse Black Beer,Foreign / Export Stout,3.0,2.5,3.0,3.0,stcules,3.0,"Almost totally black. Beige foam, quite compac...",1235916604
3,5.0,47969,10325,Sausa Pils,German Pilsener,3.5,3.0,3.0,2.5,stcules,3.0,"Golden yellow color. White, compact foam, quit...",1234725145
4,7.7,64883,1075,Cauldron DIPA,American Double / Imperial IPA,4.0,4.5,4.0,4.0,johnmichaelsen,4.5,"According to the website, the style for the Ca...",1293735206


## Data quality

Check missing values, rating ranges, and whether one user reviewed the same beer more than once.

In [5]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

print("\nRating columns describe:")
print(df[["review_appearance", "review_palette", "review_overall", "review_taste", "review_aroma"]].describe())

print("\nZero overall reviews:", (df["review_overall"] == 0).sum())
print("Zero appearance reviews:", (df["review_appearance"] == 0).sum())

Shape: (528870, 13)

Missing values:
beer_ABV              20280
beer_beerId               0
beer_brewerId             0
beer_name                 0
beer_style                0
review_appearance         0
review_palette            0
review_overall            0
review_taste              0
review_profileName      115
review_aroma              0
review_text             119
review_time               0
dtype: int64

Rating columns describe:
       review_appearance  review_palette  review_overall   review_taste  \
count      528870.000000   528870.000000   528870.000000  528870.000000   
mean            3.864522        3.758926        3.833197       3.765993   
std             0.604010        0.685335        0.709962       0.669018   
min             0.000000        1.000000        0.000000       1.000000   
25%             3.500000        3.500000        3.500000       3.500000   
50%             4.000000        4.000000        4.000000       4.000000   
75%             4.000000        4.0

### Filter invalid ratings

Keep reviews where appearance and overall are greater than 0.

In [4]:
df_filtered = df[(df["review_appearance"] > 0) & (df["review_overall"] > 0)].copy()
print("Rows before:", len(df))
print("Rows after dropping 0 scores:", len(df_filtered))

Rows before: 528870
Rows after dropping 0 scores: 528867


### Drop incomplete rows

If ABV, reviewer name, or review text is missing, drop the row.

In [6]:
df_cleaned = df_filtered.dropna().copy()
print("Rows after dropna:", len(df_cleaned))
print("\nMissing values left:")
print(df_cleaned.isna().sum())

Rows after dropna: 508355

Missing values left:
beer_ABV              0
beer_beerId           0
beer_brewerId         0
beer_name             0
beer_style            0
review_appearance     0
review_palette        0
review_overall        0
review_taste          0
review_profileName    0
review_aroma          0
review_text           0
review_time           0
dtype: int64


### One review per user–beer

Keep the highest overall score when the same profile reviewed the same beer more than once.

In [7]:
df_sorted = (
    df_cleaned
    .sort_values("review_overall", ascending=False)
    .drop_duplicates(subset=["review_profileName", "beer_beerId"], keep="first")
    .copy()
)
print("Rows after one review per user–beer:", len(df_sorted))

Rows after one review per user–beer: 503697


## Question 1 — Strongest breweries

Average ABV per brewery, using one row per beer so review volume does not bias the mean.

In [8]:
beers = df_sorted.drop_duplicates(subset=["beer_beerId"])[["beer_brewerId", "beer_ABV"]]

q1 = (
    beers.groupby("beer_brewerId")["beer_ABV"]
    .mean()
    .sort_values(ascending=False)
    .reset_index(name="mean_abv")
)
q1.head(3)

,beer_brewerId,mean_abv
0,6513,24.690000
1,736,13.500000
2,24215,12.466667


In [9]:
beer_counts = (
    df_sorted.drop_duplicates(subset=["beer_beerId"])
    .groupby("beer_brewerId")
    .size()
    .rename("n_beers")
)

q1.merge(beer_counts, on="beer_brewerId").head(3)

,beer_brewerId,mean_abv,n_beers
0,6513,24.690000,10
1,736,13.500000,3
2,24215,12.466667,3


**Answer:** Breweries `6513`, `736`, `24215` (mean ABV ~24.7, 13.5, 12.5).  
**Caveat:** #2 and #3 have only 3 beers each.

STEP 18 — Question 2 setup: year column

In [10]:
df_sorted["review_year"] = pd.to_datetime(df_sorted["review_time"], unit="s").dt.year
print(df_sorted["review_year"].min(), "to", df_sorted["review_year"].max())
print(df_sorted["review_year"].value_counts().sort_index())

1998 to 2012
review_year
1998        11
1999        10
2000        29
2001       537
2002      6723
2003     16308
2004     21026
2005     27513
2006     40367
2007     44085
2008     65926
2009     80465
2010     90482
2011    107156
2012      3059
Name: count, dtype: int64


STEP 19 — Mean overall rating by year

In [11]:
q2 = (
    df_sorted.groupby("review_year")
    .agg(
        mean_overall=("review_overall", "mean"),
        n_reviews=("review_overall", "size"),
    )
    .sort_values("mean_overall", ascending=False)
)
q2

,mean_overall,n_reviews
review_year,,
2000,4.241379,29
1998,4.045455,11
1999,4.000000,10
2001,3.963687,537
2010,3.869593,90482
2009,3.868719,80465
2005,3.845982,27513
2008,3.840571,65926
2012,3.839000,3059


STEP 20 — Question 3: which factors matter

In [12]:
q3 = (
    df_sorted.groupby("beer_beerId")[
        ["review_taste", "review_aroma", "review_appearance", "review_palette", "review_overall"]
    ]
    .mean()
)
q3.corr()

,review_taste,review_aroma,review_appearance,review_palette,review_overall
review_taste,1.000000,0.835086,0.682117,0.756190,0.823195
review_aroma,0.835086,1.000000,0.660277,0.825978,0.883863
review_appearance,0.682117,0.660277,1.000000,0.669314,0.637631
review_palette,0.756190,0.825978,0.669314,1.000000,0.766906
review_overall,0.823195,0.883863,0.637631,0.766906,1.000000


**Answer:** Aroma matters most for overall score, then taste, then palette. Appearance is weakest of the four.

Question 4 setup: beer-level scores

In [14]:
q4 = (
    df_sorted.groupby(["beer_beerId", "beer_name", "beer_style"])
    .agg(
        n_reviews=("review_overall", "size"),
        mean_overall=("review_overall", "mean"),
    )
    .reset_index()
)
print(q4["n_reviews"].describe())
q4.sort_values("mean_overall", ascending=False).head(10)

count    14990.000000
mean        33.602201
std        140.888150
min          1.000000
25%          1.000000
50%          3.000000
75%         10.000000
max       2928.000000
Name: n_reviews, dtype: float64


,beer_beerId,beer_name,beer_style,n_reviews,mean_overall
5690,32283,Chocolate Nutter,English Stout,1,5.0
11842,63618,Auld Hemp,English Bitter,1,5.0
11835,63598,One Hop Wonder Version 12,American IPA,1,5.0
11859,63673,County Line - Farmhouse Ale,Saison / Farmhouse Ale,1,5.0
3633,20326,Pilsen,German Pilsener,1,5.0
6513,37226,The Hef,Hefeweizen,1,5.0
10213,55601,Suicide By Hops,American IPA,1,5.0
3657,20480,Founders Nutty Professor Nut Stout,American Stout,1,5.0
3680,20584,Oktoberfest,MÃ¤rzen / Oktoberfest,1,5.0
11868,63735,Biere Blanche,Witbier,1,5.0


Recommend 3 beers with enough reviews

In [15]:
popular = q4[q4["n_reviews"] > 200].sort_values("mean_overall", ascending=False)
popular.head(10)

,beer_beerId,beer_name,beer_style,n_reviews,mean_overall
10317,56082,Citra DIPA,American Double / Imperial IPA,246,4.630081
3014,16814,Heady Topper,American Double / Imperial IPA,443,4.623025
8547,47658,Founders CBS Imperial Stout,American Double / Imperial Stout,618,4.597087
1171,6368,Masala Mama India Pale Ale,American IPA,627,4.491228
565,2899,Andechser Doppelbock Dunkel,Doppelbock,281,4.437722
3561,19960,Founders KBS (Kentucky Breakfast Stout),American Double / Imperial Stout,1873,4.402029
2828,15881,TrÃ¶egs Nugget Nectar,American Amber / Red Ale,1878,4.394835
1658,8954,Cantillon Saint Lamvinus,Lambic - Fruit,349,4.381089
8399,47022,Hunahpu's Imperial Stout,American Double / Imperial Stout,422,4.373223
6586,37586,Dancing Man Wheat,Hefeweizen,365,4.371233


In [16]:
import nltk
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\91917\AppData\Roaming\nltk_data...


True

In [17]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

analyser = SentimentIntensityAnalyzer()

sample = df_sorted["review_text"].head(3)
for text in sample:
    print(analyser.polarity_scores(str(text)))
    print("---")

{'neg': 0.026, 'neu': 0.816, 'pos': 0.158, 'compound': 0.9833}
---
{'neg': 0.042, 'neu': 0.717, 'pos': 0.241, 'compound': 0.9665}
---
{'neg': 0.0, 'neu': 0.816, 'pos': 0.184, 'compound': 0.9327}
---


Score every review (slow)

In [18]:
df_sorted["review_sentiment"] = df_sorted["review_text"].apply(
    lambda x: analyser.polarity_scores(str(x))["compound"]
)
print(df_sorted["review_sentiment"].describe())

count    503697.000000
mean          0.766929
std           0.393868
min          -0.996900
25%           0.780800
50%           0.930600
75%           0.974000
max           0.999900
Name: review_sentiment, dtype: float64


Favourite style from written reviews

In [19]:
q5 = (
    df_sorted.groupby("beer_style")
    .agg(
        mean_sentiment=("review_sentiment", "mean"),
        n_reviews=("review_sentiment", "size"),
        mean_overall=("review_overall", "mean"),
    )
    .sort_values("mean_sentiment", ascending=False)
)
q5.head(10)

,mean_sentiment,n_reviews,mean_overall
beer_style,,,
Quadrupel (Quad),0.857270,4808,4.053349
Dortmunder / Export Lager,0.855248,1709,4.071094
Flanders Red Ale,0.851925,2751,3.968012
Roggenbier,0.850934,137,4.054745
Braggot,0.850208,197,3.647208
American Double / Imperial Stout,0.847703,22878,4.100905
Kvass,0.846281,97,4.025773
Wheatwine,0.841312,880,3.817614
Eisbock,0.837783,194,4.082474


Text vs stars for that style

In [20]:
quad = df_sorted[df_sorted["beer_style"] == "Quadrupel (Quad)"]
quad[["review_sentiment", "review_overall"]].corr()

,review_sentiment,review_overall
review_sentiment,1.000000,0.259251
review_overall,0.259251,1.000000


## Answers

1. **Strongest breweries (mean ABV, one row per beer):** 6513 (~24.7, 10 beers), 736 (~13.5, 3 beers), 24215 (~12.5, 3 beers). #2 and #3 are small samples.

2. **Year with highest ratings:** 2000 has the highest mean but only 29 reviews. Among years with real volume, **2010** (mean ~3.87, n ≈ 90k).

3. **Important factors:** aroma (corr ~0.88) > taste (~0.82) > palette (~0.77) > appearance (~0.64), using beer-level mean ratings.

4. **Recommend 3 beers** (mean overall, >200 reviews): Citra DIPA, Heady Topper, Founders CBS Imperial Stout.

5. **Favourite style by written reviews:** Quadrupel (Quad). Sentiment vs overall stars for Quads: correlation **0.26** (positive, weak).

Source: StrataScratch / Whole Foods beer reviews.